### Retailpulse Bronze Ingestion
#### 1. Create checkpoint volume (if not exists)
In Databricks, we can create volumes to manage storage for checkpoints and other data. This command creates a volume for our bronze checkpoints if it doesn't already exist.

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.retailpulse_bronze.checkpoints")

#### 2. Verify volume and path

In [0]:
display(spark.sql("SHOW CATALOGS"))

catalog
samples
system
workspace


#### 3. Configuration
- `S3_RAW_PATH` – the source S3 directory containing the raw retail CSV files to ingest.
- `BRONZE_SCHEMA` – the Databricks catalog and schema where bronze Delta tables will be written.
- `CHECKPOINT_BASE` – the base checkpoint location used by Auto Loader, with a per-table suffix to store stream state and inferred schema.

In [0]:
S3_RAW_PATH = "s3://retailpulse-raw-data-landing/raw/retail/"
BRONZE_SCHEMA = "workspace.retailpulse_bronze" 
CHECKPOINT_BASE = "/Volumes/workspace/retailpulse_bronze/checkpoints/"

print("Configuration loaded.")
print(f"Source: {S3_RAW_PATH}")
print(f"Target schema: {BRONZE_SCHEMA}")
print(f"Checkpoints: {CHECKPOINT_BASE}")

Configuration loaded.
Source: s3://retailpulse-raw-data-landing/raw/retail/
Target schema: workspace.retailpulse_bronze
Checkpoints: /Volumes/workspace/retailpulse_bronze/checkpoints/


#### 4. Verify S3 access
List the raw S3 directory contents using `dbutils.fs.ls` and print each file name and size to confirm that Databricks can access the source data path.


In [0]:
files = dbutils.fs.ls("s3://retailpulse-raw-data-landing/raw/retail/")
for f in files:
    print(f.name, f.size)

customers.csv 2149047
orders.csv 23935043
products.csv 202079


#### 5. Bronze ingestion helper

This cell defines `ingest_to_bronze(table_name)` to ingest a specific CSV file from the raw S3 landing zone into a bronze Delta table using Auto Loader.

Data locations:
- `source_path` points to the configured raw S3 directory.
- `checkpoint_path` is a per-table checkpoint location under the bronze checkpoint volume.
- `target_table` is built from the configured bronze schema and table name.

Operations:
- `spark.readStream.format("cloudFiles")` reads CSV data from S3.
- `pathGlobFilter` limits ingestion to the matching `<table_name>.csv` file.
- `header=true` and `inferSchema=true` allow Auto Loader to interpret the CSV header and infer column types.
- The stream writes to Delta in append mode with a checkpoint and `availableNow=True` to process existing files once.
- A loop runs the helper for `customers`, `products`, and `orders`.
- `trigger(availableNow=True)` fires stream on the availability of every micro-batch of data.

In [0]:
def ingest_to_bronze(table_name):
    source_path = S3_RAW_PATH  # ← directory, not individual file
    checkpoint_path = f"{CHECKPOINT_BASE}{table_name}"
    target_table = f"{BRONZE_SCHEMA}.{table_name}"

    print(f"Ingesting {table_name} from {source_path} → {target_table} ...")

    (spark.readStream.format("cloudFiles")
                     .option("cloudFiles.format", "csv") # ← file format
                     .option("cloudFiles.schemaLocation", checkpoint_path)
                     .option("cloudFiles.useNotifications", "false") # real time ingestion not needed for this project
                     .option("pathGlobFilter", f"{table_name}.csv")  # ← filter per table
                     .option("header", "true")
                     .option("inferSchema", "true")
                     .load(source_path)
                     .writeStream # write to Delta Lake
                     .format("delta") # Target format set to Delta Lake
                     .option("checkpointLocation", checkpoint_path)
                     .outputMode("append") # Records appended to target table
                     .trigger(availableNow=True)
                     .toTable(target_table)
                     .awaitTermination() # Wait for streaming query to finish
    print(f"✅ Done: {target_table}")

# Run for all three tables
for table in ["customers", "products", "orders"]:
    ingest_to_bronze(table)

print("\n✅ All Bronze tables ingested!")

Ingesting customers from s3://retailpulse-raw-data-landing/raw/retail/ → workspace.retailpulse_bronze.customers ...
✅ Done: workspace.retailpulse_bronze.customers
Ingesting products from s3://retailpulse-raw-data-landing/raw/retail/ → workspace.retailpulse_bronze.products ...
✅ Done: workspace.retailpulse_bronze.products
Ingesting orders from s3://retailpulse-raw-data-landing/raw/retail/ → workspace.retailpulse_bronze.orders ...
✅ Done: workspace.retailpulse_bronze.orders

✅ All Bronze tables ingested!


#### 6. Verify Bronze tables

After ingesting the data, we verify that all three bronze tables were created successfully and contain the expected data. The verification cell queries each table to display row counts, column counts, and sample records.

In [0]:
for table in ["customers", "products", "orders"]:
    df = spark.table(f"{BRONZE_SCHEMA}.{table}")  # DataFrame ← ingested table, for querying
    print(f"{BRONZE_SCHEMA}.{table}: {df.count():,} rows | {len(df.columns)} columns") # Print row count and column count for a quick sanity check
    df.show(3) # Show first 3 rows for a quick sanity check
    print() # Blank line for better readability

workspace.retailpulse_bronze.customers: 100,000 rows | 6 columns
+-----------+----------------+----------+-------+-----------+-------------+
|customer_id|            name|      city|country|signup_date|_rescued_data|
+-----------+----------------+----------+-------+-----------+-------------+
|          1|   Linda Johnson| Marseille|     FR| 2021-01-31|         NULL|
|          2|Joseph Rodriguez|Manchester|     GB| 2019-02-24|         NULL|
|          3|    Aiden Garcia|     Tokyo|     JP| 2022-09-25|         NULL|
+-----------+----------------+----------+-------+-----------+-------------+
only showing top 3 rows

workspace.retailpulse_bronze.products: 10,000 rows | 6 columns
+----------+------------------+--------------+------+---------+-------------+
|product_id|              name|      category| price|stock_qty|_rescued_data|
+----------+------------------+--------------+------+---------+-------------+
|         1|  Water Bottle 093|        Sports|381.61|     4362|         NULL|
|  